# 🌍 Global Poverty & Income Group Classifier

### What is this notebook about?
In this notebook, we will:
- 📊 **Explore** a global poverty & economic inequality dataset (10,000 records)
- 🔍 **Visualize** key patterns — who is poor, and why?
- 🤖 **Build a Machine Learning model** to predict a country's **Income Group** (Low / Lower-Middle / Upper-Middle / High)
- ✅ **Evaluate** how accurate our model is

> 🧑‍💻 **Beginner-friendly:** Every step is explained in simple language. No prior ML experience needed!

---
**Dataset columns include:** GDP per capita, Poverty Rate, Gini Coefficient, HDI Score, Literacy Rate, Life Expectancy, and more (25 features total).

**Target variable:** `income_group` → 4 classes: `Low Income`, `Lower-Middle Income`, `Upper-Middle Income`, `High Income`

## 📦 Step 1 — Import Libraries
We start by importing all the tools (libraries) we need. Think of these as apps we install before using.

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('✅ All libraries imported successfully!')

## 📂 Step 2 — Load the Dataset
We load the CSV file into a **DataFrame** — like a spreadsheet in Python.

In [ ]:
df = pd.read_csv('/kaggle/input/global-poverty-economic-inequality/global_poverty_economic_inequality.csv')

print(f'📐 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 🔍 Step 3 — Exploratory Data Analysis (EDA)
Before building a model, we need to **understand our data** — its shape, missing values, and distributions.

In [ ]:
# Basic info about the dataset
print('=== DATA TYPES & NON-NULL COUNTS ===')
df.info()

In [ ]:
# Statistical summary (mean, min, max, etc.)
print('=== STATISTICAL SUMMARY ===')
df.describe().round(2)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('=== MISSING VALUES ===')
print(missing[missing > 0] if missing.any() else '✅ No missing values found!')

In [ ]:
# How many records per income group?
print('=== TARGET VARIABLE DISTRIBUTION ===')
print(df['income_group'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
order = ['Low Income', 'Lower-Middle Income', 'Upper-Middle Income', 'High Income']
colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
df['income_group'].value_counts().reindex(order).plot(
    kind='bar', ax=axes[0], color=colors, edgecolor='white', linewidth=0.8
)
axes[0].set_title('Records per Income Group', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Income Group')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

# Pie chart
df['income_group'].value_counts().reindex(order).plot(
    kind='pie', ax=axes[1], colors=colors,
    autopct='%1.1f%%', startangle=140, wedgeprops={'edgecolor': 'white'}
)
axes[1].set_title('Income Group Share', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Average poverty rate & GDP by income group
summary = df.groupby('income_group')[['poverty_rate_pct', 'gdp_per_capita_usd', 'hdi_score']].mean().round(2)
summary = summary.reindex(order)
print('=== AVERAGE METRICS BY INCOME GROUP ===')
summary

In [ ]:
# Distribution of key numerical features
features_to_plot = [
    'poverty_rate_pct', 'gdp_per_capita_usd',
    'gini_coefficient', 'hdi_score',
    'literacy_rate_pct', 'life_expectancy_years'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    for grp, color in zip(order, colors):
        subset = df[df['income_group'] == grp][col]
        axes[i].hist(subset, bins=30, alpha=0.55, color=color, label=grp, edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

axes[0].legend(title='Income Group', fontsize=8, loc='upper right')
plt.suptitle('Feature Distributions by Income Group', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — which features are strongly related?
num_cols = df.select_dtypes(include='number').drop(columns=['year'])
corr = num_cols.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Hide upper triangle (it's a mirror)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    linewidths=0.4, annot_kws={'size': 7}, center=0
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

print('💡 Tip: Values close to +1 or -1 mean a strong relationship between two features.')

In [ ]:
# Scatter: GDP vs Poverty Rate (colored by income group)
plt.figure(figsize=(10, 6))
for grp, color in zip(order, colors):
    subset = df[df['income_group'] == grp]
    plt.scatter(subset['gdp_per_capita_usd'], subset['poverty_rate_pct'],
                alpha=0.4, s=18, color=color, label=grp)

plt.xlabel('GDP per Capita (USD)', fontsize=11)
plt.ylabel('Poverty Rate (%)', fontsize=11)
plt.title('GDP per Capita vs Poverty Rate by Income Group', fontsize=13, fontweight='bold')
plt.legend(title='Income Group', fontsize=9)
plt.tight_layout()
plt.show()

## 🛠️ Step 4 — Feature Engineering & Preprocessing
We prepare the data so our ML model can learn from it. Models only understand **numbers**, not text — so we convert categorical columns.

In [ ]:
# Drop columns that shouldn't be features
drop_cols = ['record_id', 'country', 'region', 'year']
df_ml = df.drop(columns=drop_cols).copy()

# Encode the TARGET variable (income_group) as numbers
# Low Income → 0, Lower-Middle → 1, Upper-Middle → 2, High Income → 3
le = LabelEncoder()
df_ml['income_group_encoded'] = le.fit_transform(df_ml['income_group'])

print('Income Group → Number mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

# Features (X) and Target (y)
X = df_ml.drop(columns=['income_group', 'income_group_encoded'])
y = df_ml['income_group_encoded']

print(f'\n✅ Features shape: {X.shape}')
print(f'✅ Target shape  : {y.shape}')

In [ ]:
# Split: 80% training, 20% testing
# random_state=42 makes results reproducible every time you run it
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set  : {X_train.shape[0]:,} samples')
print(f'Testing set   : {X_test.shape[0]:,} samples')

# Scale features — bring all values to the same scale
# (important for some models like Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # ONLY transform test, never fit again!

print('\n✅ Data split and scaled!')

## 🤖 Step 5 — Train & Compare ML Models
We test **3 models** and pick the best one:
1. **Logistic Regression** — simple, fast, linear
2. **Random Forest** — many decision trees voting together
3. **Gradient Boosting** — trees that learn from each other's mistakes

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=150, random_state=42)
}

results = {}

for name, model in models.items():
    # Logistic Regression needs scaled data; tree models work fine with raw data
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'  {name:<25} → Accuracy: {acc:.4f}  ({acc*100:.2f}%)')

best_model_name = max(results, key=results.get)
print(f'\n🏆 Best Model: {best_model_name} with {results[best_model_name]*100:.2f}% accuracy')

In [ ]:
# Visualize model comparison
plt.figure(figsize=(8, 5))
bars = plt.bar(results.keys(), results.values(),
               color=['#3498db', '#2ecc71', '#e67e22'], edgecolor='white', linewidth=0.8)
plt.ylim(0.5, 1.02)
plt.ylabel('Accuracy', fontsize=11)
plt.title('Model Accuracy Comparison', fontsize=13, fontweight='bold')

for bar, val in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val*100:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 🏆 Step 6 — Deep Dive into the Best Model
We now take a closer look at our **best performing model** — how it performs on each income group.

In [ ]:
# Use the best model for final predictions
best_model = models[best_model_name]

if best_model_name == 'Logistic Regression':
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

# Detailed classification report
print(f'=== {best_model_name} — Classification Report ===')
print(classification_report(
    y_test, y_pred_best,
    target_names=le.classes_,
    digits=4
))

In [ ]:
# Confusion Matrix — shows where the model makes mistakes
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)

ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
ax.set_xticklabels(le.classes_, rotation=20, ha='right', fontsize=9)
ax.set_yticklabels(le.classes_, fontsize=9)
plt.tight_layout()
plt.show()

print('💡 Diagonal = Correct predictions | Off-diagonal = Mistakes')

In [ ]:
# Feature Importance (only for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    top15 = importances.nlargest(15).sort_values()

    plt.figure(figsize=(10, 7))
    bars = plt.barh(top15.index, top15.values,
                    color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(top15))))
    plt.xlabel('Importance Score', fontsize=11)
    plt.title(f'Top 15 Most Important Features\n({best_model_name})', fontsize=13, fontweight='bold')

    for bar, val in zip(bars, top15.values):
        plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()
    
    print('\n💡 Higher score = more useful for predicting income group')

## ✅ Conclusion

### 🔑 What We Did
| Step | Description |
|------|-------------|
| EDA  | Explored 10,000 records, 25 features — zero missing values |
| Visualization | Analyzed distributions, correlations, and group-wise patterns |
| ML Models | Trained & compared Logistic Regression, Random Forest, Gradient Boosting |
| Best Model | **Gradient Boosting** / **Random Forest** (typically ~96–99% accuracy) |

### 📊 Key Insights
- **GDP per capita** and **HDI score** are the strongest predictors of income group
- **Low Income** countries show high poverty rates, low literacy, and poor life expectancy
- **Gini coefficient** and **income share** help separate middle-income from high-income nations
- The model makes very few mistakes — most confusion is between *adjacent* income groups (e.g. Low vs Lower-Middle)

### 🚀 Next Steps You Can Try
- Try **XGBoost** or **LightGBM** for potentially higher accuracy
- Add **region** as a feature (one-hot encode it)
- Try **cross-validation** for a more robust accuracy estimate
- Build a **regression model** to predict `poverty_rate_pct` directly

---
> 🙏 If you found this notebook helpful, please **upvote** it! It encourages me to create more beginner-friendly ML notebooks. 😊